# LeetCode 1640 — Check Array Formation Through Concatenation bằng Python

Link đề: https://leetcode.com/problems/check-array-formation-through-concatenation/

Độ khó: **Easy** · Chủ đề: Array, Hash Table

Skeleton LeetCode:

```python
class Solution:
    def canFormArray(self, arr: list[int], pieces: list[list[int]]) -> bool:

```

Mục tiêu học:

1. Hiểu đề: ghép các mảnh `pieces` theo **thứ tự mảnh tùy ý** nhưng **không đảo phần tử trong mảnh**.
2. Khai thác điều kiện **các số đều khác nhau** để biết mỗi vị trí của `arr` thuộc đúng một mảnh.
3. Đi từ cách "quét tìm mảnh" `O(n · p)` lên **hash map phần tử đầu → mảnh** `O(n)`.
4. Hiểu thêm cách "vị trí của từng số" và cách "ghép rồi so sánh".
5. Luyện kỹ thuật hai con trỏ đơn giản: con trỏ `i` chạy trên `arr`, con trỏ `j` chạy trong một mảnh.
6. Nhận ra bẫy "ghép thành chuỗi rồi tìm chuỗi con".

## 1. Bài toán yêu cầu gì?

Cho:

- `arr`: mảng các số nguyên **khác nhau**.
- `pieces`: danh sách các mảng con; mọi số trong `pieces` (khi trải phẳng) cũng **khác nhau**.

Ta muốn tạo ra đúng `arr` bằng cách **nối** các mảng trong `pieces` theo **bất kỳ thứ tự nào**. Nhưng **không được đổi thứ tự số bên trong** mỗi `pieces[i]`.

Trả về `True` nếu làm được, ngược lại `False`.

| arr | pieces | Kết quả | Vì sao |
|---|---|---|---|
| `[15,88]` | `[[88],[15]]` | `True` | nối `[15]` rồi `[88]` |
| `[49,18,16]` | `[[16,18,49]]` | `False` | đủ số nhưng không được đảo mảnh `[16,18,49]` |
| `[91,4,64,78]` | `[[78],[4,64],[91]]` | `True` | `[91]` + `[4,64]` + `[78]` |

Constraints:

- `1 <= pieces.length <= arr.length <= 100`
- `sum(pieces[i].length) == arr.length` — tổng số phần tử của các mảnh đúng bằng độ dài `arr`.
- `1 <= arr[i], pieces[i][j] <= 100`
- Số trong `arr` khác nhau; số trong `pieces` khác nhau.

In [1]:
arr = [91, 4, 64, 78]
pieces = [[78], [4, 64], [91]]

order = [2, 1, 0]           # chọn thứ tự mảnh: pieces[2], pieces[1], pieces[0]
built = []
for idx in order:
    built += pieces[idx]
    print(f"nối pieces[{idx}] = {pieces[idx]} -> {built}")
print("Bằng arr?", built == arr)

nối pieces[2] = [91] -> [91]
nối pieces[1] = [4, 64] -> [91, 4, 64]
nối pieces[0] = [78] -> [91, 4, 64, 78]
Bằng arr? True


### Phân tích bằng tay ví dụ 3

Ta đọc `arr` từ trái sang phải:

1. Vị trí 0 là `91`. Mảnh nào **bắt đầu** bằng `91`? → `[91]`. Khớp, nhảy 1 bước.
2. Vị trí 1 là `4`. Mảnh bắt đầu bằng `4`? → `[4, 64]`. So tiếp: `arr[2] = 64` khớp. Nhảy 2 bước.
3. Vị trí 3 là `78`. Mảnh `[78]` khớp. Hết mảng → `True`.

Điểm mấu chốt: vì mọi số khác nhau, **mỗi số chỉ có thể là đầu của tối đa một mảnh**. Nên tại mỗi bước ta **không phải đoán** — chỉ có đúng một mảnh có thể đặt vào.

## 2. Nền tảng Python cần biết

### 2.1. Nối list bằng `+` và `+=`

`a + b` tạo list mới là `a` nối với `b`. `a += b` nối `b` vào cuối `a` (tại chỗ).

In [2]:
a = [91]
b = [4, 64]
print(a + b)
a += b
print(a)
print([1, 2] + [3] == [1, 2, 3])

[91, 4, 64]
[91, 4, 64]
True


### 2.2. So sánh hai list bằng `==`

Hai list bằng nhau khi **cùng độ dài** và **từng phần tử cùng vị trí bằng nhau**. Thứ tự rất quan trọng.

In [3]:
print([16, 18, 49] == [49, 18, 16])
print(sorted([16, 18, 49]) == sorted([49, 18, 16]))
print([4, 64] == [4, 64])

False
True
True


### 2.3. `dict` — tra cứu nhanh theo key

Ta sẽ lưu `first[số_đầu_tiên] = mảnh`. Kiểm tra `x in first` và đọc `first[x]` đều nhanh, trung bình `O(1)`.

`dict.get(key)` trả `None` nếu không có key (không báo lỗi).

In [4]:
pieces = [[78], [4, 64], [91]]
first = {}
for p in pieces:
    first[p[0]] = p
print(first)
print("4 in first?", 4 in first, "->", first[4])
print("64 in first?", 64 in first)
print("first.get(64) =", first.get(64))

{78: [78], 4: [4, 64], 91: [91]}
4 in first? True -> [4, 64]
64 in first? False
first.get(64) = None


### 2.4. Dict comprehension

Viết gọn vòng lặp trên: `{p[0]: p for p in pieces}` — "với mỗi mảnh `p`, key là `p[0]`, value là `p`".

In [5]:
pieces = [[78], [4, 64], [91]]
print({p[0]: p for p in pieces})

{78: [78], 4: [4, 64], 91: [91]}


### 2.5. Slicing để so một đoạn

`arr[i:i+k]` lấy đoạn dài `k` bắt đầu tại `i` (không gồm chỉ số `i+k`). Nếu vượt cuối, slicing chỉ lấy tới hết, **không báo lỗi**.

In [6]:
arr = [91, 4, 64, 78]
print(arr[1:3])
print(arr[3:6], "(vượt cuối vẫn không lỗi)")
print(arr[1:3] == [4, 64])

[4, 64]
[78] (vượt cuối vẫn không lỗi)
True


### 2.6. Vòng `while` với con trỏ tự điều khiển

Khác `for`, vòng `while` cho phép ta **nhảy** con trỏ nhiều bước một lần (`i += len(piece)`).

In [7]:
i = 0
steps = [1, 2, 1]
k = 0
while i < 4:
    print("i =", i)
    i += steps[k]
    k += 1
print("dừng tại i =", i)

i = 0
i = 1
i = 3
dừng tại i = 4


## 3. Cách trực giác — với mỗi vị trí, quét tất cả mảnh

Ý tưởng tự nhiên:

1. Đặt `i = 0`.
2. Quét tất cả `pieces`, tìm mảnh có `piece[0] == arr[i]`.
3. Không tìm được → `False`.
4. Tìm được → so từng phần tử của mảnh với `arr[i], arr[i+1], ...`. Sai một chỗ → `False`.
5. Khớp hết → `i += len(piece)`, lặp lại.

In [8]:
def can_form_scan(arr: list[int], pieces: list[list[int]]) -> bool:
    i = 0
    while i < len(arr):
        found = None
        for p in pieces:                  # quét mọi mảnh
            if p[0] == arr[i]:
                found = p
                break
        if found is None:
            return False
        for x in found:                   # so từng phần tử của mảnh
            if i >= len(arr) or arr[i] != x:
                return False
            i += 1
    return True

print(can_form_scan([15, 88], [[88], [15]]))
print(can_form_scan([49, 18, 16], [[16, 18, 49]]))
print(can_form_scan([91, 4, 64, 78], [[78], [4, 64], [91]]))

True
False
True


### Nút thắt

Mỗi lần tìm mảnh ta quét lại toàn bộ `pieces` (`p` mảnh). Số lần tìm tối đa là `p`. Tổng `O(p²)` cho phần tìm, cộng `O(n)` cho phần so.

Với `n <= 100` thì vẫn ổn, nhưng việc **quét đi quét lại** là dấu hiệu nên dùng **tra cứu**: "mảnh nào bắt đầu bằng `x`?" là câu hỏi hash map trả lời trong `O(1)` trung bình.

Còn cách "thử mọi hoán vị thứ tự mảnh" thì tệ hơn nhiều: `p!` thứ tự.

In [9]:
import math
for p in [3, 5, 10, 20]:
    print(f"p = {p:>2}: số thứ tự ghép mảnh = {math.factorial(p):,}")

p =  3: số thứ tự ghép mảnh = 6
p =  5: số thứ tự ghép mảnh = 120
p = 10: số thứ tự ghép mảnh = 3,628,800
p = 20: số thứ tự ghép mảnh = 2,432,902,008,176,640,000


## 4. Insight — hash map "số đầu tiên → mảnh"

Vì số trong `pieces` khác nhau, mỗi số là đầu của **nhiều nhất một** mảnh. Tạo:

```python
first = {p[0]: p for p in pieces}
```

Khi đứng ở `arr[i]`:

- `arr[i]` không phải đầu mảnh nào → không thể ghép (vì `arr[i]` phải là đầu một mảnh nào đó: mảnh trước đã kết thúc ngay trước `i`).
- Có mảnh `p = first[arr[i]]` → đoạn `arr[i : i+len(p)]` **bắt buộc** phải bằng `p`.

Không có lựa chọn, nên **tham lam** (greedy) là đúng.

In [10]:
arr = [91, 4, 64, 78]
pieces = [[78], [4, 64], [91]]
first = {p[0]: p for p in pieces}
i = 1
p = first[arr[i]]
print("Đang ở i =", i, "arr[i] =", arr[i], "-> mảnh", p)
print("Đoạn cần khớp:", arr[i:i + len(p)], "| khớp?", arr[i:i + len(p)] == p)

Đang ở i = 1 arr[i] = 4 -> mảnh [4, 64]
Đoạn cần khớp: [4, 64] | khớp? True


## 5. Lời giải chính — hash map + con trỏ

Giảng từng đoạn:

**Đoạn 1 — tạo bảng tra:** `first = {p[0]: p for p in pieces}`.

**Đoạn 2 — duyệt `arr` bằng `while`:** `i` là vị trí đầu của mảnh kế tiếp.

**Đoạn 3 — tra mảnh:** nếu `arr[i] not in first` → `False`.

**Đoạn 4 — so đoạn:** `arr[i:i+len(p)] != p` → `False`; khớp thì `i += len(p)`.

**Đoạn 5 — hết vòng** → `True`.

In [11]:
def can_form_hashmap(arr: list[int], pieces: list[list[int]]) -> bool:
    first = {p[0]: p for p in pieces}
    i = 0
    while i < len(arr):
        if arr[i] not in first:
            return False
        p = first[arr[i]]
        if arr[i:i + len(p)] != p:
            return False
        i += len(p)
    return True

print(can_form_hashmap([15, 88], [[88], [15]]))
print(can_form_hashmap([49, 18, 16], [[16, 18, 49]]))
print(can_form_hashmap([91, 4, 64, 78], [[78], [4, 64], [91]]))

True
False
True


## 6. TRACE từng bước

Bảng trace `arr = [91,4,64,78]`, `first = {78:[78], 4:[4,64], 91:[91]}`:

| i | arr[i] | có trong first? | mảnh p | arr[i:i+len(p)] | khớp? | i mới |
|---:|---:|:---:|---|---|:---:|---:|
| 0 | 91 | có | [91] | [91] | ✔ | 1 |
| 1 | 4 | có | [4,64] | [4,64] | ✔ | 3 |
| 3 | 78 | có | [78] | [78] | ✔ | 4 |

`i = 4 = len(arr)` → dừng, trả `True`.

Ví dụ 2 (`arr = [49,18,16]`, `first = {16: [16,18,49]}`): ở `i = 0`, `49` **không** là đầu mảnh nào → `False` ngay.

In [12]:
def trace_can_form(arr: list[int], pieces: list[list[int]]) -> bool:
    first = {p[0]: p for p in pieces}
    print("first =", first)
    i = 0
    while i < len(arr):
        x = arr[i]
        if x not in first:
            print(f"i={i}: arr[i]={x} không là đầu mảnh nào -> False")
            return False
        p = first[x]
        seg = arr[i:i + len(p)]
        ok = seg == p
        print(f"i={i}: arr[i]={x} -> mảnh {p}, đoạn arr = {seg}, khớp? {ok}")
        if not ok:
            return False
        i += len(p)
    print(f"i={i} = len(arr) -> True")
    return True

trace_can_form([91, 4, 64, 78], [[78], [4, 64], [91]])
print()
trace_can_form([49, 18, 16], [[16, 18, 49]])

first = {78: [78], 4: [4, 64], 91: [91]}
i=0: arr[i]=91 -> mảnh [91], đoạn arr = [91], khớp? True
i=1: arr[i]=4 -> mảnh [4, 64], đoạn arr = [4, 64], khớp? True
i=3: arr[i]=78 -> mảnh [78], đoạn arr = [78], khớp? True
i=4 = len(arr) -> True

first = {16: [16, 18, 49]}
i=0: arr[i]=49 không là đầu mảnh nào -> False


False

In [13]:
# một ví dụ sai ở giữa mảnh: mảnh [1, 3] nhưng arr có 1 rồi 2
trace_can_form([1, 2, 3], [[2], [1, 3]])

first = {2: [2], 1: [1, 3]}
i=0: arr[i]=1 -> mảnh [1, 3], đoạn arr = [1, 2], khớp? False


False

## 7. Code nộp LeetCode

In [14]:
class Solution:
    def canFormArray(self, arr: list[int], pieces: list[list[int]]) -> bool:
        first = {p[0]: p for p in pieces}
        i = 0
        while i < len(arr):
            p = first.get(arr[i])
            if p is None or arr[i:i + len(p)] != p:
                return False
            i += len(p)
        return True

sol = Solution()
assert sol.canFormArray([15, 88], [[88], [15]]) is True
assert sol.canFormArray([49, 18, 16], [[16, 18, 49]]) is False
assert sol.canFormArray([91, 4, 64, 78], [[78], [4, 64], [91]]) is True
print("3 ví dụ của đề PASS")

3 ví dụ của đề PASS


`p is None or ...`: toán tử `or` **ngắt mạch** (short-circuit). Nếu `p is None` là `True`, Python không tính vế sau — nhờ vậy không bị lỗi `len(None)`.

In [15]:
p = None
print(p is None or len(p) > 0)   # không lỗi vì vế phải không được tính

True


## 8. Các lời giải khác

### 8.1. Bảng vị trí: mỗi mảnh phải nằm liền nhau trong `arr`

Đảo góc nhìn: thay vì đi trên `arr` tìm mảnh, ta đi trên **từng mảnh** và hỏi "mảnh này có xuất hiện liền mạch trong `arr` không?".

- `pos[v]` = vị trí của `v` trong `arr`.
- Mảnh `p` hợp lệ nếu mọi `p[j]` có trong `arr` và `pos[p[j]] == pos[p[0]] + j`.

Vì tổng độ dài các mảnh bằng `len(arr)` và mọi số khác nhau, nếu mọi mảnh đều nằm liền mạch thì chúng phủ kín `arr` không chồng nhau → ghép được.

In [16]:
def can_form_positions(arr: list[int], pieces: list[list[int]]) -> bool:
    pos = {v: i for i, v in enumerate(arr)}
    for p in pieces:
        if p[0] not in pos:
            return False
        start = pos[p[0]]
        for j, v in enumerate(p):
            if pos.get(v) != start + j:
                return False
    return True

print(can_form_positions([15, 88], [[88], [15]]))
print(can_form_positions([49, 18, 16], [[16, 18, 49]]))
print(can_form_positions([91, 4, 64, 78], [[78], [4, 64], [91]]))

True
False
True


### 8.2. Xếp mảnh theo vị trí rồi ghép và so sánh

1. Sắp các mảnh theo vị trí của phần tử đầu trong `arr` (mảnh có đầu không thuộc `arr` → `False`).
2. Nối các mảnh theo thứ tự đó.
3. So với `arr`.

Đây là cách "dựng lại rồi so", rất dễ tin là đúng.

In [17]:
def can_form_rebuild(arr: list[int], pieces: list[list[int]]) -> bool:
    pos = {v: i for i, v in enumerate(arr)}
    if any(p[0] not in pos for p in pieces):
        return False
    ordered = sorted(pieces, key=lambda p: pos[p[0]])
    built = []
    for p in ordered:
        built += p
    return built == arr

print(can_form_rebuild([15, 88], [[88], [15]]))
print(can_form_rebuild([49, 18, 16], [[16, 18, 49]]))
print(can_form_rebuild([91, 4, 64, 78], [[78], [4, 64], [91]]))

True
False
True


`any(điều_kiện for ...)` trả `True` nếu **ít nhất một** điều kiện đúng (ngược với `all`).

In [18]:
print(any(x > 5 for x in [1, 7, 3]))
print(any(x > 9 for x in [1, 7, 3]))

True
False


### 8.3. Bảng so sánh các cách

| Cách | Ý tưởng | Thời gian | Bộ nhớ | Nhận xét |
|---|---|---|---|---|
| Thử mọi thứ tự mảnh | hoán vị | `O(p! · n)` | `O(n)` | không khả thi |
| Quét tìm mảnh | mỗi vị trí quét `pieces` | `O(p² + n)` | `O(1)` | dễ hiểu, không cần dict |
| Hash map đầu mảnh | `first[x]` → mảnh | `O(n)` | `O(p)` | **nên dùng** |
| Bảng vị trí | mỗi mảnh liền mạch | `O(n)` | `O(n)` | góc nhìn ngược lại |
| Sắp mảnh rồi ghép | dựng lại `arr` | `O(n + p log p)` | `O(n)` | dễ chứng minh đúng |

(`n = len(arr)`, `p = len(pieces)`; vì `p <= n` các cách `O(n)` là tốt nhất.)

## 9. Vì sao lời giải chính đúng?

**Invariant:** ở đầu mỗi vòng `while`, đoạn `arr[0:i]` đã được ghép **chính xác** bằng một số mảnh, và `i` là vị trí bắt đầu của mảnh kế tiếp.

- Ban đầu `i = 0`, đoạn rỗng — đúng.
- Tại `i`, phần tử `arr[i]` phải là **đầu** của mảnh kế tiếp. Vì số trong `pieces` khác nhau, chỉ **một** mảnh có thể bắt đầu bằng `arr[i]`. Không có → không cách nào ghép → `False` là đúng.
- Có mảnh `p` nhưng `arr[i:i+len(p)] != p` → mảnh bắt buộc này không khớp → `False` là đúng.
- Khớp → `i += len(p)`, invariant giữ.

Khi `i == len(arr)` thì toàn bộ `arr` đã được ghép. Còn có mảnh nào thừa không? Không: tổng độ dài mảnh bằng `len(arr)` và mảnh đã dùng không lặp lại (vì số trong `arr` khác nhau, mỗi số đầu chỉ gặp một lần), nên mọi mảnh đều đã dùng đúng một lần.

## 10. Độ phức tạp

Gọi `n = len(arr)`, `p = len(pieces)`.

- Tạo `first`: `O(p)`.
- Vòng `while`: mỗi phần tử của `arr` được so đúng một lần trong slicing → tổng `O(n)`.
- **Time: `O(n)`** (vì `p <= n`). Tra dict là `O(1)` trung bình.
- **Space: `O(p)`** cho `first` (value chỉ là tham chiếu tới mảnh có sẵn), cộng `O(độ dài mảnh)` tạm thời cho slicing.

## 11. Bộ test

1. 3 ví dụ của đề.
2. Biên: `arr` một phần tử.
3. Một mảnh duy nhất bằng chính `arr`.
4. Phần tử đầu có mảnh nhưng mảnh bị lệch ở giữa.
5. Hai ca "ngoài constraints" (mảnh vượt cuối, số không có trong `arr`) để thấy code vẫn an toàn.
6. Đối chiếu 4 cách trên dữ liệu ngẫu nhiên (cả trường hợp ghép được và không).

In [19]:
def check(name, actual, expected):
    assert actual == expected, (name, actual, expected)
    print(f"{name:<30} PASS -> {actual}")

f = can_form_hashmap
check("đề 1", f([15, 88], [[88], [15]]), True)
check("đề 2", f([49, 18, 16], [[16, 18, 49]]), False)
check("đề 3", f([91, 4, 64, 78], [[78], [4, 64], [91]]), True)
check("1 phần tử", f([7], [[7]]), True)
check("1 mảnh = arr", f([1, 2, 3], [[1, 2, 3]]), True)
check("lệch giữa mảnh", f([1, 2, 3], [[2], [1, 3]]), False)
check("mảnh vượt cuối (ngoài constr.)", f([3, 1, 2], [[2, 5], [3, 1]]), False)
check("số lạ (ngoài constraints)", f([1, 3, 5, 7], [[2, 4, 6, 8]]), False)

đề 1                           PASS -> True
đề 2                           PASS -> False
đề 3                           PASS -> True
1 phần tử                      PASS -> True
1 mảnh = arr                   PASS -> True
lệch giữa mảnh                 PASS -> False
mảnh vượt cuối (ngoài constr.) PASS -> False
số lạ (ngoài constraints)      PASS -> False


In [20]:
import random

rng = random.Random(1640)

def random_case():
    n = rng.randint(1, 10)
    arr = rng.sample(range(1, 30), n)
    cuts = sorted(rng.sample(range(1, n), rng.randint(0, n - 1))) if n > 1 else []
    bounds = [0] + cuts + [n]
    pieces = [arr[bounds[k]:bounds[k + 1]] for k in range(len(bounds) - 1)]
    if rng.random() < 0.5:                       # làm hỏng ngẫu nhiên
        k = rng.randrange(len(pieces))
        pieces[k] = pieces[k][::-1]
    rng.shuffle(pieces)
    return arr, pieces

for _ in range(500):
    arr, pieces = random_case()
    ref = can_form_rebuild(arr, pieces)
    assert can_form_scan(arr, pieces) == ref
    assert can_form_hashmap(arr, pieces) == ref
    assert can_form_positions(arr, pieces) == ref
    assert Solution().canFormArray(arr, pieces) == ref
print("5 cách khớp nhau trên 500 input ngẫu nhiên")

5 cách khớp nhau trên 500 input ngẫu nhiên


`rng.sample(range(1, 30), n)` lấy `n` số **khác nhau** — đúng ràng buộc đề. `pieces[k][::-1]` đảo mảnh để tạo ca sai (nếu mảnh dài hơn 1). `rng.shuffle` xáo thứ tự mảnh tại chỗ.

## 12. Lỗi người mới hay gặp

### Lỗi 1: Chỉ kiểm tra "cùng tập số"

Sai: so `sorted(arr) == sorted(phẳng(pieces))` — bỏ qua thứ tự trong mảnh. Ví dụ 2 sẽ ra `True` sai.

In [21]:
def wrong_same_set(arr, pieces):
    flat = [x for p in pieces for x in p]
    return sorted(arr) == sorted(flat)

print("sai :", wrong_same_set([49, 18, 16], [[16, 18, 49]]))
print("đúng:", can_form_hashmap([49, 18, 16], [[16, 18, 49]]))

sai : True
đúng: False


`[x for p in pieces for x in p]` là list comprehension **hai vòng**: đọc như `for p in pieces: for x in p: lấy x` — dùng để trải phẳng list lồng.

### Lỗi 2: Chỉ kiểm tra phần tử đầu của mảnh, quên so phần còn lại

In [22]:
def wrong_first_only(arr, pieces):
    first = {p[0]: p for p in pieces}
    i = 0
    while i < len(arr):
        if arr[i] not in first:
            return False
        i += len(first[arr[i]])
    return True

print("sai :", wrong_first_only([1, 2, 3, 4], [[1, 4], [3, 2]]))
print("đúng:", can_form_hashmap([1, 2, 3, 4], [[1, 4], [3, 2]]))

sai : True
đúng: False


### Lỗi 3: Ghép thành chuỗi rồi dùng `in`

Ý tưởng "nối arr thành chuỗi, kiểm tra mỗi mảnh là chuỗi con" sai vì chữ số dính nhau: `[1, 23]` → `"123"` chứa `"12"` dù `[1, 2]` không phải đoạn của `arr`.

In [23]:
def wrong_string(arr, pieces):
    s = "".join(map(str, arr))
    return all("".join(map(str, p)) in s for p in pieces)

print("sai :", wrong_string([1, 23], [[1, 2], [3]]), "(thậm chí số 2, 3 không có trong arr)")
print("đúng:", can_form_hashmap([1, 23], [[1, 2], [3]]))

sai : True (thậm chí số 2, 3 không có trong arr)
đúng: False


`map(str, arr)` áp `str` cho từng phần tử; `"".join(...)` nối các chuỗi lại không có ký tự ngăn cách.

### Lỗi 4: Dùng `for i in range(len(arr))` rồi cố "nhảy" i

Gán `i += len(p)` bên trong `for` **không** làm vòng for nhảy, vì `range` tự cấp giá trị `i` mới mỗi vòng.

In [24]:
for i in range(5):
    print("i =", i, end=" | ")
    i += 3          # vô tác dụng với vòng lặp kế tiếp
print()
print("=> Muốn nhảy con trỏ, dùng while.")

i = 0 | i = 1 | i = 2 | i = 3 | i = 4 | 
=> Muốn nhảy con trỏ, dùng while.


### Lỗi 5: Truy cập `first[arr[i]]` khi key không tồn tại

`first[key]` với key không có sẽ báo `KeyError`. Kiểm tra `in` trước, hoặc dùng `.get()`.

In [25]:
first = {16: [16, 18, 49]}
try:
    first[49]
except KeyError as e:
    print("KeyError:", e)
print("an toàn:", first.get(49))

KeyError: 49
an toàn: None


## 13. Mẹo nhận dạng pattern

- "Các số khác nhau" + "mỗi phần tử thuộc đúng một nhóm" → dùng **hash map từ phần tử tới nhóm/vị trí**.
- Khi mỗi bước chỉ có **một lựa chọn hợp lệ** → greedy, không cần thử mọi hoán vị.
- Con trỏ cần nhảy theo độ dài khối → **`while` + tự tăng con trỏ**.
- "Ghép mảnh theo thứ tự tùy ý" → sắp mảnh theo vị trí trong đích, hoặc tra mảnh theo phần tử đầu.

Bài liên quan: 1662 Check If Two String Arrays are Equivalent, 1048 Longest String Chain (tra cứu theo key), 1436 Destination City (map đầu → cuối).

## 14. Biến thể

### Biến thể 1: Trả về thứ tự chỉ số mảnh để ghép (hoặc `[]` nếu không được)

In [26]:
def formation_order(arr, pieces):
    first = {p[0]: k for k, p in enumerate(pieces)}
    order, i = [], 0
    while i < len(arr):
        k = first.get(arr[i])
        if k is None or arr[i:i + len(pieces[k])] != pieces[k]:
            return []
        order.append(k)
        i += len(pieces[k])
    return order

print(formation_order([91, 4, 64, 78], [[78], [4, 64], [91]]))
print(formation_order([49, 18, 16], [[16, 18, 49]]))

[2, 1, 0]
[]


### Biến thể 2: Cho phép đảo ngược từng mảnh

Mỗi mảnh có thể dùng xuôi hoặc ngược. Ta lưu cả hai đầu vào bảng tra.

In [27]:
def can_form_reversible(arr, pieces):
    first = {}
    for p in pieces:
        first[p[0]] = p
        first[p[-1]] = p[::-1]
    i = 0
    while i < len(arr):
        p = first.get(arr[i])
        if p is None or arr[i:i + len(p)] != p:
            return False
        i += len(p)
    return True

print(can_form_reversible([49, 18, 16], [[16, 18, 49]]))
print(can_form_reversible([1, 2, 3], [[2], [1, 3]]))

True
False


### Biến thể 3: Mảnh có thể dư (không bắt buộc dùng hết)

Bỏ ràng buộc `sum(len) == len(arr)`. Cách hash map vẫn chạy được nguyên vẹn vì nó chỉ đi theo `arr`; còn cách "bảng vị trí" 8.1 sẽ sai vì mảnh dư có số không nằm trong `arr`.

In [28]:
arr, pieces = [1, 2], [[1], [2], [9]]
print("hash map  :", can_form_hashmap(arr, pieces))
print("positions :", can_form_positions(arr, pieces), "(sai khi cho phép mảnh dư)")

hash map  : True
positions : False (sai khi cho phép mảnh dư)


## 15. Bài tập tự luyện

Các cell TODO chỉ định nghĩa hàm, không được gọi khi chạy notebook.

### Bài 1 — Tính tay

`arr = [5, 1, 2, 9, 7]`, `pieces = [[9, 7], [5], [1, 2]]`. Viết bảng trace `i, arr[i], mảnh, khớp?`. Kết quả?

### Bài 2 — Tạo bảng tra đầu mảnh

In [29]:
def exercise_first_map(pieces: list[list[int]]) -> dict:
    # TODO: trả về dict {phần tử đầu: mảnh}
    pass

# assert exercise_first_map([[78], [4, 64], [91]]) == {78: [78], 4: [4, 64], 91: [91]}

### Bài 3 — Viết lại lời giải chính

In [30]:
def exercise_can_form(arr: list[int], pieces: list[list[int]]) -> bool:
    # TODO 1: first = ...
    # TODO 2: while i < len(arr): tra mảnh, so đoạn, nhảy i
    pass

# assert exercise_can_form([91, 4, 64, 78], [[78], [4, 64], [91]]) is True
# assert exercise_can_form([49, 18, 16], [[16, 18, 49]]) is False

### Bài 4 — Cài đặt cách bảng vị trí

In [31]:
def exercise_positions(arr: list[int], pieces: list[list[int]]) -> bool:
    # TODO: pos = {giá trị: chỉ số}; mỗi mảnh phải có pos liên tiếp
    pass

# assert exercise_positions([15, 88], [[88], [15]]) is True
# assert exercise_positions([1, 2, 3], [[2], [1, 3]]) is False

### Bài 5 — Biến thể với chuỗi

Cho `target` (chuỗi) và `parts` (list chuỗi, các ký tự đầu của các phần **khác nhau**). Kiểm tra có ghép được `target` bằng cách nối mỗi phần đúng một lần theo thứ tự tùy ý không.

In [32]:
def exercise_strings(target: str, parts: list[str]) -> bool:
    # TODO: tương tự first map nhưng key là ký tự đầu
    pass

# assert exercise_strings("abcde", ["cd", "ab", "e"]) is True
# assert exercise_strings("abcde", ["ce", "ab", "d"]) is False

### Bài 6 — Mini challenge: số mảnh tối thiểu bị thiếu

Cho `arr` và `pieces` (mảnh có thể không phủ hết `arr`, nhưng mảnh nào được dùng thì phải khớp nguyên vẹn). Mỗi vị trí không được phủ bởi mảnh phải tự "vá" bằng một mảnh 1 phần tử mới. Trả về số mảnh vá ít nhất; nếu có mảnh đặt vào bị lệch thì trả `-1`.

In [33]:
def exercise_patches(arr: list[int], pieces: list[list[int]]) -> int:
    # TODO: đi trên arr; nếu arr[i] là đầu mảnh thì phải khớp, không thì vá 1
    pass

# assert exercise_patches([1, 2, 3, 4], [[2, 3]]) == 2
# assert exercise_patches([1, 2, 3], [[2, 4]]) == -1

## 16. Gợi ý 3 mức

**Bài 1**
- Mức 1: tạo `first = {9:[9,7], 5:[5], 1:[1,2]}`.
- Mức 2: bắt đầu `i = 0`, `arr[0] = 5`.
- Mức 3: `i: 0 → 1 → 3 → 5`, mọi bước khớp.

**Bài 2**
- Mức 1: key là `p[0]`.
- Mức 2: dict comprehension.
- Mức 3: `{p[0]: p for p in pieces}`.

**Bài 3**
- Mức 1: dùng `while`, không dùng `for range`.
- Mức 2: `p = first.get(arr[i])`; `None` thì `False`.
- Mức 3: so `arr[i:i+len(p)] != p`, rồi `i += len(p)`.

**Bài 4**
- Mức 1: `pos = {v: i for i, v in enumerate(arr)}`.
- Mức 2: `start = pos[p[0]]`.
- Mức 3: với `j, v in enumerate(p)` kiểm tra `pos.get(v) == start + j`.

**Bài 5**
- Mức 1: giống bài chính, đơn vị là ký tự.
- Mức 2: `first = {s[0]: s for s in parts}`.
- Mức 3: so `target[i:i+len(s)] == s`, chú ý thêm `sum(map(len, parts)) == len(target)`.

**Bài 6**
- Mức 1: các số khác nhau → mảnh bắt đầu bằng `arr[i]` là duy nhất.
- Mức 2: nếu `arr[i]` không là đầu mảnh: `patch += 1`, `i += 1`.
- Mức 3: nếu là đầu mảnh mà đoạn không khớp → `-1`.

## 17. Đáp án tham khảo

**Bài 1:** `i=0 → [5]` khớp, `i=1 → [1,2]` khớp, `i=3 → [9,7]` khớp, `i=5` dừng → `True`.

In [34]:
def answer_first_map(pieces):
    return {p[0]: p for p in pieces}

def answer_can_form(arr, pieces):
    first = {p[0]: p for p in pieces}
    i = 0
    while i < len(arr):
        p = first.get(arr[i])
        if p is None or arr[i:i + len(p)] != p:
            return False
        i += len(p)
    return True

def answer_positions(arr, pieces):
    pos = {v: i for i, v in enumerate(arr)}
    for p in pieces:
        if p[0] not in pos:
            return False
        start = pos[p[0]]
        for j, v in enumerate(p):
            if pos.get(v) != start + j:
                return False
    return True

def answer_strings(target, parts):
    if sum(map(len, parts)) != len(target):
        return False
    first = {s[0]: s for s in parts}
    i = 0
    while i < len(target):
        s = first.get(target[i])
        if s is None or target[i:i + len(s)] != s:
            return False
        i += len(s)
    return True

def answer_patches(arr, pieces):
    first = {p[0]: p for p in pieces}
    i, patch = 0, 0
    while i < len(arr):
        p = first.get(arr[i])
        if p is None:
            patch += 1
            i += 1
        elif arr[i:i + len(p)] != p:
            return -1
        else:
            i += len(p)
    return patch

assert answer_can_form([5, 1, 2, 9, 7], [[9, 7], [5], [1, 2]]) is True   # Bài 1
assert answer_first_map([[78], [4, 64], [91]]) == {78: [78], 4: [4, 64], 91: [91]}
assert answer_can_form([91, 4, 64, 78], [[78], [4, 64], [91]]) is True
assert answer_can_form([49, 18, 16], [[16, 18, 49]]) is False
assert answer_positions([15, 88], [[88], [15]]) is True
assert answer_positions([1, 2, 3], [[2], [1, 3]]) is False
assert answer_strings("abcde", ["cd", "ab", "e"]) is True
assert answer_strings("abcde", ["ce", "ab", "d"]) is False
assert answer_patches([1, 2, 3, 4], [[2, 3]]) == 2
assert answer_patches([1, 2, 3], [[2, 4]]) == -1
print("Tất cả đáp án PASS")

Tất cả đáp án PASS


## 18. Template nhớ nhanh + checklist submit

```python
class Solution:
    def canFormArray(self, arr: list[int], pieces: list[list[int]]) -> bool:
        first = {p[0]: p for p in pieces}       # 1) đầu mảnh -> mảnh
        i = 0
        while i < len(arr):                     # 2) con trỏ trên arr
            p = first.get(arr[i])
            if p is None or arr[i:i + len(p)] != p:
                return False                    # 3) không có mảnh / lệch
            i += len(p)                         # 4) nhảy qua cả mảnh
        return True
```

Câu thần chú: **"Số khác nhau → mảnh bắt đầu bằng arr[i] là duy nhất → tra dict, so đoạn, nhảy."**

Checklist trước khi submit:

- [ ] Dùng `while` để nhảy con trỏ (không `for range`)?
- [ ] So **toàn bộ** mảnh, không chỉ phần tử đầu?
- [ ] Xử lý `arr[i]` không phải đầu mảnh nào (không `KeyError`)?
- [ ] Không so "cùng tập số" thay cho "đúng thứ tự"?
- [ ] Không ghép chuỗi số rồi dùng `in`?
- [ ] Đã thử ca lệch giữa mảnh như `arr=[1,2,3], pieces=[[2],[1,3]]`?